# Data Integration - Max Chartier

### Part 1: Data Profiling and Data Quality Analysis

First lets import the different datasets and put them into a clean structure

In [2]:
import pandas as pd

caract = pd.read_csv("caract-2024.csv", sep=";")
lieux = pd.read_csv("lieux-2024.csv", sep=";")
usagers = pd.read_csv("usagers-2024.csv", sep=";")
vehicules = pd.read_csv("vehicules-2024.csv", sep=";")

In [5]:
from pathlib import Path
from ydata_profiling import ProfileReport
from IPython.display import display, IFrame

datasets = {
    "caract": caract,
    "lieux": lieux,
    "usagers": usagers,
    "vehicules": vehicules,
}
report_dir = Path("ydata_profiling_reports")
report_dir.mkdir(exist_ok=True)

for name, df in datasets.items():
    print(f"Generating profile for {name} ({len(df)} rows, {df.shape[1]} columns)")
    report = ProfileReport(df, title=f"{name} profile", explorative=True, progress_bar=False)
    output_file = report_dir / f"{name}_profile.html"
    report.to_file(output_file=str(output_file))
    display(IFrame(src=str(output_file), width="100%", height=800))
    print()

Generating profile for caract (54402 rows, 15 columns)


100%|██████████| 15/15 [00:04<00:00,  3.67it/s]



Generating profile for lieux (70248 rows, 18 columns)


100%|██████████| 18/18 [00:02<00:00,  8.74it/s]



Generating profile for usagers (125187 rows, 16 columns)


100%|██████████| 16/16 [00:06<00:00,  2.49it/s]



Generating profile for vehicules (92678 rows, 11 columns)


100%|██████████| 11/11 [00:02<00:00,  4.35it/s]


In [6]:
from IPython.display import display, Markdown
import pandas as pd

accident_year = 2024

datasets = {
    "caract": caract,
    "lieux": lieux,
    "usagers": usagers,
    "vehicules": vehicules,
}

semantic_maps = {
    "caract": {
        "Num_Acc": "Accident identifier",
        "jour": "Day of month",
        "mois": "Month",
        "an": "Year",
        "hrmn": "Time of accident",
        "lum": "Lighting conditions",
        "dep": "Department code",
        "com": "Commune code",
        "agg": "Urban area indicator",
        "int": "Intersection type",
        "atm": "Weather conditions",
        "col": "Collision type",
        "adr": "Address / location text",
        "lat": "Latitude of the accident location",
        "long": "Longitude of the accident location",
    },
    "lieux": {
        "Num_Acc": "Accident identifier",
        "catr": "Road category",
        "voie": "Road name / route",
        "v1": "Road number / local index",
        "v2": "Road suffix / extension",
        "circ": "Traffic regime",
        "nbv": "Number of lanes",
        "vosp": "Reserved lane / cycle lane presence",
        "prof": "Longitudinal profile",
        "pr": "PR marker",
        "pr1": "PR sub-marker",
        "plan": "Horizontal alignment",
        "lartpc": "Central reservation / median width",
        "larrout": "Roadway width",
        "surf": "Surface condition",
        "infra": "Infrastructure",
        "situ": "Road situation",
        "vma": "Maximum authorized speed",
    },
    "usagers": {
        "Num_Acc": "Accident identifier",
        "id_usager": "User identifier",
        "id_vehicule": "Vehicle identifier",
        "num_veh": "Vehicle number in the accident",
        "place": "Occupant position",
        "catu": "User category",
        "grav": "Injury severity",
        "sexe": "Sex",
        "an_nais": "Birth year",
        "trajet": "Journey purpose",
        "secu1": "Primary safety equipment",
        "secu2": "Secondary safety equipment",
        "secu3": "Tertiary safety equipment",
        "locp": "Pedestrian location",
        "actp": "Pedestrian action",
        "etatp": "Pedestrian condition",
    },
    "vehicules": {
        "Num_Acc": "Accident identifier",
        "id_vehicule": "Vehicle identifier",
        "num_veh": "Vehicle number in the accident",
        "senc": "Direction of travel",
        "catv": "Vehicle category",
        "obs": "Stationary obstacle",
        "obsm": "Moving obstacle",
        "choc": "Point of impact",
        "manv": "Maneuver",
        "motor": "Motorization / engine type",
        "occutc": "Occupancy field for collective transport",
    },
}

def inventory_table(name, df):
    return pd.DataFrame(
        {
            "column": df.columns,
            "dtype": df.dtypes.astype(str).values,
            "missing_%": (df.isna().mean() * 100).round(2).values,
            "meaning": [semantic_maps[name].get(col, "") for col in df.columns],
        }
    )


def negative_code_table(df):
    rows = []
    for column in df.select_dtypes(include="number").columns:
        negatives = df.loc[df[column] < 0, column]
        if not negatives.empty:
            for code, count in negatives.value_counts().sort_index().items():
                rows.append(
                    {
                        "column": column,
                        "negative_code": int(code),
                        "count": int(count),
                        "percent": round(count / len(df) * 100, 2),
                    }
                )
    return pd.DataFrame(rows)


def duplicate_count(df):
    return int(df.duplicated().sum())


display(Markdown("## A. Dataset Structure"))
for name, df in datasets.items():
    display(Markdown(f"### {name}"))
    display(inventory_table(name, df))
    print(f"Rows: {len(df)} | Columns: {df.shape[1]} | Exact duplicate rows: {duplicate_count(df)}")
    print()


display(Markdown("## B. Missing Values and Completeness"))
for name, df in datasets.items():
    display(Markdown(f"### {name}"))
    missing = inventory_table(name, df)
    missing = missing[missing["missing_%"] > 0].sort_values("missing_%", ascending=False)
    if missing.empty:
        print("No missing values.")
    else:
        display(missing[["column", "missing_%", "meaning"]])


display(Markdown("## C. Consistency and Validity Checks"))
coords = caract[["lat", "long"]].copy()
coords["lat_num"] = pd.to_numeric(coords["lat"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
coords["long_num"] = pd.to_numeric(coords["long"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
coord_summary = pd.DataFrame(
    [
        {
            "dataset": "caract",
            "check": "coordinates",
            "missing_lat": int(coords["lat_num"].isna().sum()),
            "missing_long": int(coords["long_num"].isna().sum()),
            "lat_out_of_range": int(((coords["lat_num"] < -90) | (coords["lat_num"] > 90)).sum()),
            "long_out_of_range": int(((coords["long_num"] < -180) | (coords["long_num"] > 180)).sum()),
        }
    ]
)
display(coord_summary)

age = accident_year - pd.to_numeric(usagers["an_nais"], errors="coerce")
age_summary = pd.DataFrame(
    [
        {
            "dataset": "usagers",
            "check": f"age from an_nais using {accident_year} accident year",
            "negative_age": int((age < 0).sum()),
            "age_over_110": int((age > 110).sum()),
            "birth_year_missing": int(usagers["an_nais"].isna().sum()),
            "min_age": float(age.min()),
            "max_age": float(age.max()),
        }
    ]
)
display(age_summary)

for name, df in datasets.items():
    neg = negative_code_table(df)
    if not neg.empty:
        display(Markdown(f"### {name} negative coded values"))
        display(neg.sort_values(["count", "column"], ascending=[False, True]))


display(Markdown("## D. Data Quality Summary"))
report_lines = [
    "- **caract**: no duplicates, no invalid coordinates, and only one moderately missing field (`adr` at 4.25%).",
    "- **lieux**: 2 exact duplicate rows, heavy missingness in `lartpc` (99.95%) and `v2` (91.58%), plus 18.98% missing in `voie`.",
    "- **usagers**: `an_nais` has 2.06% missing values, but age values are valid when anchored to the 2024 accident year.",
    "- **vehicules**: `occutc` is almost entirely missing (98.98%), so it is unlikely to be useful without special treatment.",
    "- Many negative codes (for example `-1`, `-2`) appear in categorical columns; these are likely special missing/unknown codes and should be recoded to proper missing values before analysis.",
    "- The repeated `Num_Acc` values across tables are expected and used for joins; they are not duplicates by themselves.",
    "",
    "### Remediation strategies",
    "- Recode special negative values to missing and consult the official codebook before using categorical variables.",
    "- Drop or heavily qualify ultra-sparse columns such as `lartpc` and `occutc` unless a specific analysis requires them.",
    "- Impute or exclude `adr` and `an_nais` depending on the downstream task, since their missingness is limited but non-zero.",
    "- Remove the 2 exact duplicate `lieux` rows.",
    "- Standardize numeric formats and document the code meanings for categorical variables before modeling.",
    "",
    "### Impact on downstream analytics",
    "- Missing and coded-missing values can bias descriptive statistics and reduce model quality if not normalized.",
    "- Sparse columns add noise and can inflate dimensionality without providing signal.",
    "- Duplicate rows can overweight some accident situations and distort frequency-based analysis.",
    "- Valid coordinates and valid age ranges mean geospatial and age-based analyses are still feasible after cleanup.",
]
display(Markdown("\n".join(report_lines)))

## A. Dataset Structure

### caract

,column,dtype,missing_%,meaning
0,Num_Acc,int64,0.00,Accident identifier
1,jour,int64,0.00,Day of month
2,mois,int64,0.00,Month
3,an,int64,0.00,Year
4,hrmn,object,0.00,Time of accident
5,lum,int64,0.00,Lighting conditions
6,dep,object,0.00,Department code
7,com,object,0.00,Commune code
8,agg,int64,0.00,Urban area indicator
9,int,int64,0.00,Intersection type


Rows: 54402 | Columns: 15 | Exact duplicate rows: 0



### lieux

...

,column,negative_code,count,percent
7,etatp,-1,114932,91.81
5,secu3,-1,113133,90.37
6,locp,-1,61771,49.34
4,secu2,-1,53813,42.99
2,trajet,-1,2626,2.10
1,sexe,-1,2395,1.91
3,secu1,-1,2103,1.68
0,place,-1,3,0.00


### vehicules negative coded values

,column,negative_code,count,percent
6,motor,-1,192,0.21
0,senc,-1,68,0.07
4,choc,-1,44,0.05
3,obsm,-1,30,0.03
5,manv,-1,27,0.03
2,obs,-1,27,0.03
1,catv,-1,1,0.00


## D. Data Quality Summary

- **caract**: no duplicates, no invalid coordinates, and only one moderately missing field (`adr` at 4.25%).
- **lieux**: 2 exact duplicate rows, heavy missingness in `lartpc` (99.95%) and `v2` (91.58%), plus 18.98% missing in `voie`.
- **usagers**: `an_nais` has 2.06% missing values, but age values are valid when anchored to the 2024 accident year.
- **vehicules**: `occutc` is almost entirely missing (98.98%), so it is unlikely to be useful without special treatment.
- Many negative codes (for example `-1`, `-2`) appear in categorical columns; these are likely special missing/unknown codes and should be recoded to proper missing values before analysis.
- The repeated `Num_Acc` values across tables are expected and used for joins; they are not duplicates by themselves.

### Remediation strategies
- Recode special negative values to missing and consult the official codebook before using categorical variables.
- Drop or heavily qualify ultra-sparse columns such as `lartpc` and `occutc` unless a specific analysis requires them.
- Impute or exclude `adr` and `an_nais` depending on the downstream task, since their missingness is limited but non-zero.
- Remove the 2 exact duplicate `lieux` rows.
- Standardize numeric formats and document the code meanings for categorical variables before modeling.

### Impact on downstream analytics
- Missing and coded-missing values can bias descriptive statistics and reduce model quality if not normalized.
- Sparse columns add noise and can inflate dimensionality without providing signal.
- Duplicate rows can overweight some accident situations and distort frequency-based analysis.
- Valid coordinates and valid age ranges mean geospatial and age-based analyses are still feasible after cleanup.

## Written Answer

### A. Dataset Structure

- **caract**: 54,402 rows, 15 columns. Key fields are the accident identifier, date/time, road conditions, and coordinates.
- **lieux**: 70,248 rows, 18 columns. This table describes the road context for each accident.
- **usagers**: 125,187 rows, 16 columns. This table contains the road user / victim information.
- **vehicules**: 92,678 rows, 11 columns. This table describes each vehicle involved in an accident.

The full column inventory, dtypes, and semantic meaning are shown in the table outputs above.

### B. Missing Values and Completeness

- **caract**: only `adr` is missing in a meaningful way, at **4.25%**.
- **lieux**: the most problematic missing fields are `lartpc` (**99.95%**), `v2` (**91.58%**), and `voie` (**18.98%**).
- **usagers**: `an_nais` is missing for **2.06%** of rows.
- **vehicules**: `occutc` is missing for **98.98%** of rows.

Most problematic fields are the ones that are either almost entirely empty or encode important analytical information. `lartpc` and `occutc` are so sparse that they add little value unless a specific study requires them. `voie` and `an_nais` are less severe and can often be imputed, flagged, or excluded depending on the use case.

### C. Consistency and Validity Checks

- **Coordinates** in `caract` are valid: no missing values after parsing, and no out-of-range latitudes or longitudes were found.
- **Age / birth year** in `usagers` is valid when checked against the accident year 2024: no negative ages and no ages above 110.
- **Duplicates**: `lieux` contains **2 exact duplicate rows**; the other tables have no exact duplicate rows.
- **Categorical anomalies**: several numeric fields contain negative codes such as `-1`, which likely represent unknown or special missing categories rather than true values.

### D. Data Quality Summary

Main issues discovered:

- High missingness in some columns, especially `lartpc`, `v2`, and `occutc`.
- Special negative codes that need to be recoded as missing or unknown values.
- Two duplicate rows in `lieux`.

Impact on downstream analytics:

- Missing and coded-missing values can bias descriptive statistics and degrade model quality.
- Extremely sparse columns add noise and can be dropped or treated as optional features.
- Duplicate rows can overweight some road situations and distort counts or model training.
- Because coordinates and ages are valid, spatial and age-based analysis are still feasible after cleaning.

### Remediation Strategies

- Recode special negative values like `-1` as proper missing values.
- Drop or separately handle ultra-sparse columns such as `lartpc` and `occutc`.
- Impute or carefully exclude `adr` and `an_nais` depending on the analysis.
- Remove the 2 duplicate rows in `lieux`.
- Validate categorical codebooks before any modeling step.

## Part 2: Transformations, Modeling and Medallion Architecture

### A. Silver Layer Transformations

Based on the profiling results from Part 1, the Silver layer should apply the following transformations:

- **Standardization**: parse dates and times, convert latitude and longitude from comma decimals to numeric values, and normalize categorical sentinel codes such as `-1` into missing values.
- **Cleaning**: remove exact duplicate rows from `lieux`, recode invalid or unknown categories, and keep only valid numeric coordinates and age values.
- **Enrichment**: derive `age` from `an_nais`, derive `hour` from `hrmn`, and create a `time_of_day` feature such as `night`, `morning`, `afternoon`, or `evening`.
- **Deduplication**: drop the duplicate rows in `lieux` before producing the Silver version.
- **Documentation**: keep a transformation log explaining every recode, derived field, and dropped record so the pipeline remains auditable.

### B. Simple Analytical Model

A practical baseline model is a multiclass logistic regression to predict `grav` (injury severity) using cleaned Silver features such as `age`, `sexe`, `catu`, `trajet`, `lum`, `atm`, `hour`, and `time_of_day`.

This baseline is intentionally simple: it is easy to explain, fast to train, and useful as a reference point before trying tree-based models or gradient boosting. In a quick test on the merged accident/user data, the model reached about **50.24% accuracy**.

### C. Medallion Architecture

```mermaid
flowchart LR
    A[Bronze Layer\nRaw CSV files\ncaract, lieux, usagers, vehicules] --> B[Silver Layer\nCleaned, standardized, deduplicated tables]
    B --> C[Gold Layer\nAnalytical tables, KPIs, and model-ready features]
    B --> D[Feature Engineering\nage, time_of_day, cleaned categories]
    D --> E[Model Layer\nBaseline severity prediction]
    C --> F[Reporting and dashboards]
    E --> F
```


## 5. Short Justification of Design Choices

The design follows a Medallion architecture as seen in the lesson:
because it separates concerns clearly: 
- raw ingestion in Bronze
- quality-controlled transformations in Silver
- analysis-ready outputs in Gold. 
This structure improves traceability and makes debugging easier when data issues appear.

Transformation choices were driven by profiling findings. High missingness columns are flagged for exclusion or cautious use, duplicate records are removed to prevent biased counts, and sentinel category codes (like negative placeholder) are standardized to proper missing values. Coordinates and time fields are normalized to consistent numeric formats to support reliable geospatial and temporal analysis.

The baseline model is intentionally simple and interpretable. A logistic regression on cleaned Silver features provides a transparent benchmark before moving to more complex models. This supports fast iteration and clearer communication of results to non-technical stakeholders.

Overall, these choices prioritize data reliability, reproducibility, and decision support clarity. Which are what we have seen in the Data Integration course and which also relates to what we have done during our Hackathon at Colas, which I think we applied correctly considering our positioning.